# ⚡ Boosting
**ISLP Chapter 8 · Pattern Recognition for the Rest of Us**

> While Random Forests train trees independently and average them, Boosting trains trees *sequentially* — each tree learns from the mistakes of the previous one. This iterative error-correction makes Boosting one of the highest-performing ML approaches.

### What you'll learn
- AdaBoost — upweighting misclassified points
- Gradient Boosting — fitting trees to residuals
- XGBoost — the industry-standard implementation
- Key hyperparameters: n_estimators, learning_rate, max_depth
- Comparing Boosting vs Random Forests

### Dataset: California Housing (regression) + Heart disease (classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
!pip install xgboost -q
import xgboost as xgb
print("✓ XGBoost:", xgb.__version__)

In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df_house = data.frame
X = df_house.drop('MedHouseVal', axis=1)
y = df_house['MedHouseVal']
print(f"California Housing: {df_house.shape}")
df_house.head()

In [ ]:
# Load California Housing
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing(as_frame=True)
df_house = data.frame
X = df_house.drop('MedHouseVal', axis=1)
y = df_house['MedHouseVal']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"California Housing: {X.shape[0]:,} census blocks")
print(f"Predicting: median house value (in $100,000s)")
print(f"Features: {list(X.columns)}")

## ⚡ Part 1 — How Gradient Boosting Works

**Core idea:** Train trees sequentially. Each tree fits the *residuals* (errors) from all previous trees.

**Step by step:**
1. Initialize with a simple prediction (e.g. mean of Y)
2. Compute residuals: rᵢ = yᵢ - ŷᵢ
3. Fit a small tree to the residuals
4. Update: ŷᵢ ← ŷᵢ + λ × (tree prediction)  (λ = learning rate, shrinks each step)
5. Repeat 2-4 for B iterations

The **learning rate** (shrinkage) λ is critical — small λ requires more trees but generalizes better.

In [ ]:
# Show boosting building up — predictions improve with each tree
from sklearn.ensemble import GradientBoostingRegressor

stages = [1, 5, 25, 100, 300]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

# Use a small 1D slice for visualization
X_viz = X_tr[['MedInc']].copy()
y_viz = y_tr.values

for ax, n in zip(axes, stages):
    gbm = GradientBoostingRegressor(n_estimators=n, learning_rate=0.1,
                                     max_depth=3, random_state=42)
    gbm.fit(X_viz, y_viz)
    
    xi = np.linspace(X_viz['MedInc'].min(), X_viz['MedInc'].max(), 200).reshape(-1,1)
    yi_pred = gbm.predict(xi)
    
    ax.scatter(X_viz['MedInc'], y_viz, alpha=0.1, s=5, color='#888')
    ax.plot(xi, yi_pred, color='#e85d20', lw=2)
    mse = mean_squared_error(y_viz, gbm.predict(X_viz))
    ax.set_title(f'n={n}\nRMSE={np.sqrt(mse):.3f}', fontsize=10)
    ax.set_xlabel('Median Income')
    if n==1: ax.set_ylabel('House Value')

plt.suptitle('Gradient Boosting — Predictions Improve with More Trees', y=1.05, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Compare: sklearn GBM vs XGBoost
from sklearn.metrics import mean_squared_error as mse

models = {
    'GBM (sklearn)': GradientBoostingRegressor(n_estimators=300, learning_rate=0.1,
                                                max_depth=4, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=300, learning_rate=0.1,
                                  max_depth=4, random_state=42,
                                  eval_metric='rmse', verbosity=0),
}

print(f"{'Model':<20} {'Train RMSE':>12} {'Test RMSE':>12}")
print("-"*45)
for name, model in models.items():
    model.fit(X_tr, y_tr)
    tr_rmse = np.sqrt(mse(y_tr, model.predict(X_tr)))
    te_rmse = np.sqrt(mse(y_te, model.predict(X_te)))
    print(f"{name:<20} {tr_rmse:>12.4f} {te_rmse:>12.4f}")
print("\n📌 XGBoost uses second-order gradients, regularization, and parallel computing")
print("   — often faster and slightly better than sklearn's GBM")

In [ ]:
# Effect of learning rate and n_estimators (the key tradeoff)
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.3]
n_estimators   = [10, 50, 100, 200, 500]
results = np.zeros((len(learning_rates), len(n_estimators)))

for i, lr in enumerate(learning_rates):
    for j, n in enumerate(n_estimators):
        gbm = GradientBoostingRegressor(n_estimators=n, learning_rate=lr,
                                         max_depth=3, random_state=42)
        gbm.fit(X_tr, y_tr)
        results[i,j] = np.sqrt(mse(y_te, gbm.predict(X_te)))

fig, ax = plt.subplots(figsize=(10, 4))
for i, lr in enumerate(learning_rates):
    ax.plot(n_estimators, results[i], 'o-', lw=1.8, markersize=5,
            label=f'lr={lr}')
ax.set_xlabel('n_estimators')
ax.set_ylabel('Test RMSE')
ax.set_title('Learning Rate × n_estimators Tradeoff\n(small lr requires many trees but generalizes better)')
ax.legend(title='Learning rate', fontsize=9)
plt.tight_layout()
plt.show()
print("\n📌 Small learning rate (0.01) + many trees often outperforms large lr + few trees")
print("   But small lr is slower to train — XGBoost's early stopping solves this")

In [ ]:
# Feature importance (XGBoost)
xgb_model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05,
                               max_depth=4, random_state=42, verbosity=0)
xgb_model.fit(X_tr, y_tr)

imp = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e85d20' if v > imp.quantile(0.6) else '#1e5fa8' for v in imp]
ax.barh(imp.index, imp.values, color=colors, edgecolor='white')
ax.set_title('XGBoost Feature Importance — California Housing')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()
final_rmse = np.sqrt(mse(y_te, xgb_model.predict(X_te)))
print(f"\nFinal XGBoost Test RMSE: {final_rmse:.4f}")
print("📌 MedInc (median income) dominates — wealthy neighborhoods = higher house values")

In [ ]:
# @title 📝 Quiz — Boosting
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the core difference between Random Forest and Gradient Boosting?
# @markdown **Q2:** What does the learning rate control in gradient boosting?
# @markdown **Q3:** Why does a very high learning rate (e.g. 0.9) often perform poorly?
# @markdown **Q4:** What is the advantage of XGBoost over sklearn's GradientBoosting?
# @markdown **Q5:** If your boosting model overfits, name two hyperparameters you would adjust

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Boosting"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
